# Practical 3: TF-IDF Vectorization

**Concept:** Bag-of-Words only counts words, so common words dominate the numbers. **TF-IDF (Term Frequency – Inverse Document Frequency)** fixes this by weighing each term with how **important** it is. A word is important if it appears often in a *single* document, but is *rare* across the whole corpus.

$$
TF(t, d) = \frac{\text{count of term } t \text{ in document } d}{\text{total words in document } d}
$$

$$
IDF(t) = \log\frac{\text{total documents } N}{\text{number of documents containing } t}
$$

$$
TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
$$

In this notebook we:
1. Build TF, IDF and TF-IDF **manually**
2. Use scikit-learn's `TfidfVectorizer`
3. Compare manual vs library results
4. Evaluate the vectors using cosine similarity

## Import Required Libraries

In [6]:
%pip install pandas numpy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/tejas/.local/share/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import math

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Original Corpus
We use a small corpus of NLP-related sentences so the math is easy to verify by hand.

In [8]:
corpus = [
    "machine learning transform artificial intelligence",
    "machine learning use neural network",
    "deep learning subset machine learning"
]

print("Corpus")
print("-" * 50)

for i, doc in enumerate(corpus, start=1):
    print(f"Document {i}: {doc}")

Corpus
--------------------------------------------------
Document 1: machine learning transform artificial intelligence
Document 2: machine learning use neural network
Document 3: deep learning subset machine learning


## Tokenization & Vocabulary
We tokenize each document by splitting on spaces and build a sorted vocabulary of unique words. The vocabulary determines the columns of our vectors.

In [9]:
# Tokenize each document
tokenized = [doc.split() for doc in corpus]

# Build a sorted unique vocabulary
vocabulary = sorted(set(word for doc in tokenized for word in doc))

print("Vocabulary")
print("-" * 50)
print(vocabulary)
print(f"\nVocabulary size: {len(vocabulary)}")

Vocabulary
--------------------------------------------------
['artificial', 'deep', 'intelligence', 'learning', 'machine', 'network', 'neural', 'subset', 'transform', 'use']

Vocabulary size: 10


## 1. Term Frequency (TF)
TF measures how frequently a term appears in a document.
Formula: `(number of times term t appears in doc d) / (total words in doc d)`

In [10]:
# Build the TF matrix manually
tf_matrix = []

for doc in tokenized:
    tf_vector = []
    for word in vocabulary:
        # Term count in doc / Total words in doc
        tf = doc.count(word) / len(doc)
        tf_vector.append(round(tf, 4))
    tf_matrix.append(tf_vector)

print("TF Matrix (rows = documents, columns = vocabulary)")
print("-" * 50)

for row in tf_matrix:
    print(row)

TF Matrix (rows = documents, columns = vocabulary)
--------------------------------------------------
[0.2, 0.0, 0.2, 0.2, 0.2, 0.0, 0.0, 0.0, 0.2, 0.0]
[0.0, 0.0, 0.0, 0.2, 0.2, 0.2, 0.2, 0.0, 0.0, 0.2]
[0.0, 0.2, 0.0, 0.4, 0.2, 0.0, 0.0, 0.2, 0.0, 0.0]


## 2. Inverse Document Frequency (IDF)
IDF measures how **rare** (and therefore informative) a term is across the whole corpus. Words present in every document (like "machine" or "learning") get a low/zero IDF.
Formula: `log(Total documents / Number of documents containing term t)`

In [11]:
N = len(corpus)
idf_vector = []

for word in vocabulary:
    # Count how many documents contain this word
    df = sum(1 for doc in tokenized if word in doc)

    # Standard IDF calculation
    idf = math.log10(N / df)
    idf_vector.append(round(idf, 4))

print("IDF Vector")
print("-" * 50)
print(idf_vector)

print("\nWord -> IDF")
for word, value in zip(vocabulary, idf_vector):
    print(f"  {word:<15} {value}")

IDF Vector
--------------------------------------------------
[0.4771, 0.4771, 0.4771, 0.0, 0.0, 0.4771, 0.4771, 0.4771, 0.4771, 0.4771]

Word -> IDF
  artificial      0.4771
  deep            0.4771
  intelligence    0.4771
  learning        0.0
  machine         0.0
  network         0.4771
  neural          0.4771
  subset          0.4771
  transform       0.4771
  use             0.4771


## 3. Manual TF-IDF Matrix
Multiply TF by IDF element-wise for every (document, word) pair.

In [12]:
tfidf_matrix = []

for tf_vector in tf_matrix:
    tfidf_vector = []
    for i in range(len(vocabulary)):
        # TF * IDF
        tfidf = tf_vector[i] * idf_vector[i]
        tfidf_vector.append(round(tfidf, 4))
    tfidf_matrix.append(tfidf_vector)

print("Manual TF-IDF Matrix")
print("-" * 50)

for row in tfidf_matrix:
    print(row)

Manual TF-IDF Matrix
--------------------------------------------------
[0.0954, 0.0, 0.0954, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0954, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0954, 0.0954, 0.0, 0.0, 0.0954]
[0.0, 0.0954, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0954, 0.0, 0.0]


## 4. Scikit-Learn TfidfVectorizer
scikit-learn uses a **smoothed** IDF (`+1` in the numerator and denominator, as if an extra document contained every term) and **L2-normalizes** each vector by default. So the exact values differ from our manual calculation, but the concept is identical — common words get pushed down, rare words get boosted.

In [13]:
# Create and fit the vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

print("Vocabulary:")
print(vectorizer.get_feature_names_out())
print("\nTF-IDF Matrix (scikit-learn):")
print(X.toarray())

Vocabulary:
['artificial' 'deep' 'intelligence' 'learning' 'machine' 'network'
 'neural' 'subset' 'transform' 'use']

TF-IDF Matrix (scikit-learn):
[[0.52004008 0.         0.52004008 0.30714405 0.30714405 0.
  0.         0.         0.52004008 0.        ]
 [0.         0.         0.         0.30714405 0.30714405 0.52004008
  0.52004008 0.         0.         0.52004008]
 [0.         0.51680194 0.         0.61046311 0.30523155 0.
  0.         0.51680194 0.         0.        ]]


## 5. Viewing as a DataFrame
A pandas DataFrame makes it easy to read the matrix — each row is a document and each column is a vocabulary word.

In [14]:
# Convert to a readable DataFrame
df_tfidf = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("TF-IDF Matrix as DataFrame")
print("-" * 50)
print(df_tfidf.round(4))

TF-IDF Matrix as DataFrame
--------------------------------------------------
       artificial    deep  intelligence  learning  machine  network  neural  \
Doc 1        0.52  0.0000          0.52    0.3071   0.3071     0.00    0.00   
Doc 2        0.00  0.0000          0.00    0.3071   0.3071     0.52    0.52   
Doc 3        0.00  0.5168          0.00    0.6105   0.3052     0.00    0.00   

       subset  transform   use  
Doc 1  0.0000       0.52  0.00  
Doc 2  0.0000       0.00  0.52  
Doc 3  0.5168       0.00  0.00  


## 6. Top Terms per Document
For each document we can find which words carry the most weight. These are the words that best summarize that document.

In [15]:
print("Top 3 Terms per Document")
print("-" * 50)

for doc_index in range(X.shape[0]):
    # Get this document's vector
    vector = X[doc_index].toarray()[0]

    # Sort word indices by TF-IDF weight (descending)
    top_indices = np.argsort(vector)[::-1][:3]

    top_terms = [
        (vectorizer.get_feature_names_out()[i], round(vector[i], 4))
        for i in top_indices if vector[i] > 0
    ]

    print(f"Document {doc_index + 1}: {top_terms}")

Top 3 Terms per Document
--------------------------------------------------
Document 1: [('transform', np.float64(0.52)), ('artificial', np.float64(0.52)), ('intelligence', np.float64(0.52))]
Document 2: [('use', np.float64(0.52)), ('neural', np.float64(0.52)), ('network', np.float64(0.52))]
Document 3: [('learning', np.float64(0.6105)), ('subset', np.float64(0.5168)), ('deep', np.float64(0.5168))]


## 7. Evaluation with Cosine Similarity
A classic use of TF-IDF vectors is measuring how **similar** two documents are via cosine similarity. Documents 1 and 2 both discuss "machine learning" but use different words, so their similarity tells us how related they are.

In [16]:
# Cosine similarity matrix between all documents
similarity_matrix = cosine_similarity(X)

print("Cosine Similarity Matrix")
print("-" * 50)
print(np.round(similarity_matrix, 4))
print()
print(f"Similarity between Doc 1 and Doc 2: {similarity_matrix[0][1]:.4f}")
print(f"Similarity between Doc 1 and Doc 3: {similarity_matrix[0][2]:.4f}")

Cosine Similarity Matrix
--------------------------------------------------
[[1.     0.1887 0.2813]
 [0.1887 1.     0.2813]
 [0.2813 0.2813 1.    ]]

Similarity between Doc 1 and Doc 2: 0.1887
Similarity between Doc 1 and Doc 3: 0.2813


## 8. Transforming a New Document
The fitted vectorizer can also convert a **new**, unseen document into the same feature space. Words that were never seen in training are simply ignored.

In [17]:
new_doc = ["neural network is a subset of machine learning"]

# Only transform (no fit) so the vocabulary stays identical
new_vector = vectorizer.transform(new_doc)

print("New Document:", new_doc[0])
print("\nNew Document Vector:")
print(new_vector.toarray())
print("\nFeature Names:")
print(vectorizer.get_feature_names_out())
print(f"\nSimilarity to Doc 2: {cosine_similarity(new_vector, X)[0][1]:.4f}")

New Document: neural network is a subset of machine learning

New Document Vector:
[[0.         0.         0.         0.30714405 0.30714405 0.52004008
  0.52004008 0.52004008 0.         0.        ]]

Feature Names:
['artificial' 'deep' 'intelligence' 'learning' 'machine' 'network'
 'neural' 'subset' 'transform' 'use']

Similarity to Doc 2: 0.7296
